In [1]:
from io import BytesIO
from pathlib import Path
from urllib.request import urlretrieve

import pyarrow.parquet as pq
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader
from torchvision import models, transforms

torch.set_num_threads(2)
torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,) * 3, (0.5,) * 3)])

def load_cifar(split, size, labels=None):
    path = Path(f"cifar10-{split}.parquet")
    if not path.exists():
        url = f"https://huggingface.co/datasets/uoft-cs/cifar10/resolve/main/plain_text/{split}-00000-of-00001.parquet"
        partial = path.with_suffix(".part")
        urlretrieve(url, partial)
        partial.replace(path)
    examples = []
    counts = {}
    for batch in pq.ParquetFile(path).iter_batches(batch_size=512):
        for row in batch.to_pylist():
            label = row["label"]
            if labels is not None:
                if label not in labels or counts.get(label, 0) == size:
                    continue
                counts[label] = counts.get(label, 0) + 1
                label = labels.index(label)
            with Image.open(BytesIO(row["img"]["bytes"])) as image:
                examples.append((transform(image.convert("RGB")), label))
            if len(examples) == size * (len(labels) if labels is not None else 1):
                return examples
    raise ValueError("CIFAR-10 contains fewer examples than requested")

train = load_cifar("train", 600, (3, 5))
test = load_cifar("test", 250, (3, 5))

def fit(model, optimizer, epochs=4):
    batches = DataLoader(train, batch_size=64, shuffle=True, generator=torch.Generator().manual_seed(7))
    for epoch in range(epochs):
        model.train()
        for x, y in batches:
            optimizer.zero_grad()
            loss = nn.functional.cross_entropy(model(x.to(device)), y.to(device))
            loss.backward()
            optimizer.step()
    model.eval()
    with torch.no_grad():
        predictions = torch.cat([model(x.to(device)).argmax(1).cpu() for x, _ in evaluation])
    return predictions

evaluation = DataLoader(test, batch_size=128)
labels = torch.cat([y for _, y in evaluation])

for architecture in (models.resnet18, models.mobilenet_v3_small):
    torch.manual_seed(7)
    model = architecture(weights=None, num_classes=2).to(device)
    predictions = fit(model, torch.optim.Adam(model.parameters()), epochs=4)
    print(f"{architecture.__name__:20} test accuracy: {(predictions == labels).float().mean():.1%}")

resnet18             test accuracy: 56.2%
mobilenet_v3_small   test accuracy: 50.0%
